# Parallel 08 — PP 流水线并行：按层切，气泡是唯一的敌人

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/parallel/08-pipeline-parallel.ipynb)

- **日期**：2026-10-02
- **定位**：并行基础线第四篇，补齐 TP/DP/PP 三件套。TP 切层内、DP 切数据、PP 切层间 —— 三种通信长相不同，组合原则在这一篇收口。
- **环境**：CPU + gloo，2 进程验证机制；调度模拟是纯计算。

配套笔记：[08-pipeline-parallel.md](08-pipeline-parallel.md)


## 0. 因果链定位

TP 把单层切开有上限（一层就那么宽），DP/ZeRO 不碰层间结构。模型深到几十上百层时，最后一招是按层切：$ p $ 个 stage 摊到 $ p $ 个设备，激活逐段往后流。

新问题随之诞生：第一个 micro-batch 流过 $ p $ 段之前，后面的设备都在干等 —— **气泡**。PP 的全部设计都在和气泡作战，而武器只有一个：把 batch 切成 $ m $ 个 micro-batch 轮流喂。这一篇先把气泡公式模拟验证，再手写一个真 2-stage 流水线。


## 1. 气泡公式与调度模拟

设前向、反向各占 1 个时间单位（真实比例约 1:2，公式结构不变）。GPipe 调度（全部前向、再全部反向）下：

- 理想时间（无气泡）：$ 2m $ （每段做 $ m $ 前向 + $ m $ 反向）
- 实际 makespan：$ 2(m + p - 1) $ （填满 $ p-1 $ 步 + 稳态 + 排空 $ p-1 $ 步）

$$ \text{bubble 比例} = \frac{p-1}{m+p-1} $$

下面用离散调度模拟器验证这个公式，并对比 GPipe 与 1F1B 的**激活占用**（两者气泡相同，差在显存）。


In [ ]:
def simulate(p, m, mode):
    """离散调度：每段同一时刻只能做一件事，前/反向各 1 单位。
    返回 (makespan, 第 0 段的最大 in-flight 数)。"""
    # 任务状态：F[s][i], B[s][i] 的完成时刻
    F = [[None] * m for _ in range(p)]
    B = [[None] * m for _ in range(p)]
    stage_free = [0] * p
    inflight = [0] * p                      # 每段已前向未反向的 micro 数
    inflight0_max = 0

    def ready_F(s, i):
        if F[s][i] is not None: return None
        # 1F1B 的核心约束：第 s 段 in-flight 上限 p-s（这样反向能及时跟上）
        if mode == '1f1b' and inflight[s] >= p - s: return None
        t = stage_free[s]
        if s > 0:
            if F[s-1][i] is None: return None
            t = max(t, F[s-1][i])
        if i > 0 and F[s][i-1] is None: return None
        if i > 0: t = max(t, F[s][i-1])
        return t
    def ready_B(s, i):
        if B[s][i] is not None: return None
        t = stage_free[s]
        if F[s][i] is None: return None
        t = max(t, F[s][i])
        if s < p - 1:
            if B[s+1][i] is None: return None
            t = max(t, B[s+1][i])
        if i > 0 and B[s][i-1] is None: return None
        if i > 0: t = max(t, B[s][i-1])
        return t

    done = 0
    while done < 2 * p * m:
        # 选下一个可执行任务：GPipe 优先前向；1F1B 有反向先做反向
        cands = []
        for s in range(p):
            for i in range(m):
                tF, tB = ready_F(s, i), ready_B(s, i)
                if tF is not None: cands.append((tF, 0 if mode == 'gpipe' else 1, s, i, 'F'))
                if tB is not None: cands.append((tB, 1 if mode == 'gpipe' else 0, s, i, 'B'))
        cands.sort(key=lambda c: (c[0], c[1], c[3]))
        t, _, s, i, kind = cands[0]
        if kind == 'F':
            F[s][i] = t + 1
            inflight[s] += 1
            if s == 0: inflight0_max = max(inflight0_max, inflight[s])
        else:
            B[s][i] = t + 1
            inflight[s] -= 1
        stage_free[s] = t + 1
        done += 1
    makespan = max(max(row) for row in B)
    return makespan, inflight0_max

for mode in ['gpipe', '1f1b']:
    ms, inflight = simulate(p=4, m=8, mode=mode)
    bubble = (ms - 2 * 8) / ms
    print(f'{mode:6s}: makespan {ms}（公式 2*(m+p-1)={2*(8+4-1)}） bubble {bubble:.2f}（公式 (p-1)/(m+p-1)={3/11:.2f}） stage0 最大 in-flight {inflight}（m={8}）')
print('\n读法：两种调度气泡相同（公式都成立）；差别在 stage0 要存几份激活 —— GPipe 存全部 m 份，1F1B 只存约 p 份。')


## 2. 手写 2-stage 流水线：send/recv 与梯度回流

真实现的骨架只有四步：前一段算完 `send` 激活、后一段 `recv` 后接着算；反向时后一段把激活的梯度 `send` 回来、前一段 `recv` 后继续 `backward`。配对顺序写反就是死锁 —— PP 版的 hang 和 DDP 的 hang 是两种死法。


In [ ]:
import copy, os, torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn.functional as Fn

def run(rank, world_size):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29505'
    dist.init_process_group('gloo', rank=rank, world_size=world_size)
    torch.manual_seed(0)

    # 参考实现：4 层 MLP 单进程版（两 rank 用同一种子建同一份，对拍用）
    ref = torch.nn.Sequential(torch.nn.Linear(8, 16), torch.nn.ReLU(),
                              torch.nn.Linear(16, 16), torch.nn.ReLU(),
                              torch.nn.Linear(16, 4))
    x = torch.randn(4, 8); y = torch.randint(0, 4, (4,))
    loss_ref = Fn.cross_entropy(ref(x), y)
    loss_ref.backward()
    ref_grads = [p.grad.clone() for p in ref.parameters()]

    if rank == 0:
        stage = torch.nn.Sequential(copy.deepcopy(ref[0]), copy.deepcopy(ref[1]))  # 前两层（深拷贝：不能与对拍用的 ref 共享参数对象）
        a = stage(x)
        dist.send(a.detach(), dst=1)                          # 激活送下一段
        grad_a = torch.empty_like(a)
        dist.recv(grad_a, src=1)                              # 接回梯度
        a.backward(grad_a)
        mine = [p.grad for p in stage.parameters()]
        names = ['stage0.layer0', 'stage0.layer1']
    else:
        stage = torch.nn.Sequential(copy.deepcopy(ref[2]), copy.deepcopy(ref[3]), copy.deepcopy(ref[4]))  # 后三层
        a_in = torch.empty(4, 16)
        dist.recv(a_in, src=0)
        a_in.requires_grad_(True)
        loss = Fn.cross_entropy(stage(a_in), y)
        loss.backward()
        dist.send(a_in.grad, dst=0)                           # 梯度送回上一段
        mine = [p.grad for p in stage.parameters()]
        names = ['stage1.layer0', 'stage1.layer1', 'stage1.layer2']
        print('loss 对拍: 流水线', round(loss.item(), 6), 'vs 单卡', round(loss_ref.item(), 6))

    ref_slice = ref_grads[:2] if rank == 0 else ref_grads[2:]
    for nm, g, gr in zip(names, mine, ref_slice):
        print(f'rank{rank} {nm} 权重梯度最大差: {(g - gr).abs().max().item():.2e}')
    dist.destroy_process_group()

if __name__ == '__main__':
    mp.spawn(run, args=(2,), nprocs=2, join=True)


## 3. 调试清单

- **症状：死锁** —— send/recv 配对顺序反了（两边都在等对方先说话）。
  **修复**：PP 的 hang 先查这里，和 DDP 的"少进一次 collective"完全不同；收发顺序按调度表逐对对齐。
- **症状：梯度整体放大约 m 倍** —— micro-batch 梯度没除以 m。
  **修复**：和 Research 01 的 accumulation 同款 bug —— PP 里每个 micro 的 loss 都要先除 $ m $ 再 backward 累积。
- **症状：流水线节奏被最慢段锁死，气泡公式失效** —— stage 切分不均，某一段计算量远大于其他段。
  **修复**：切层前先估每段的 FLOPs 再切，不要按层数平均分；embedding 层尤其容易成为隐藏瓶颈。
- **症状：退化成多个模型在各训各的** —— PP×DP 组合时，同一 stage 在不同 DP 副本间初始化不一致。
  **修复**：不同 stage 之间权重不同是正常的，但同一 stage 跨 DP 副本必须同种子初始化（或广播同步）。

## 小结

- PP 的核心交易：气泡 $ (p-1)/(m+p-1) $ 换激活显存 $ \propto m $ ；1F1B 不动气泡、把 in-flight 压到约 $ p $ 份，是标配调度。
- 通信长相决定位置：PP 只在边界点对点传激活（量小、能忍延迟）→ 跨节点；TP 是 all-reduce（频繁、小消息）→ 节点内。并行组合的总原则：**通信越频繁的维度，放在带宽越高、延迟越低的层级**。
- 至此并行三件套齐活：DP 切数据、TP 切层内、PP 切层间，ZeRO/FSDP 切存储 —— 面试里"给定模型和集群，怎么组合并行策略"的问题，答案就是从这四个维度各算一遍账。